# Notebook 16 — Prototype Memory Bank Aging

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 15 updated the prototype bank by learning a new drift prototype.

Notebook 16 turns that updated bank into a managed memory system:

- prototype age,
- usage frequency,
- stability score,
- residual contribution,
- pruning candidates,
- refreshed / retained prototypes.

Constraint view:
> adaptive prototype memory needs both recall and forgetting.

## Goals

1. Load Notebook 15 recovery outputs when available.
2. Load updated prototype bank when available.
3. Track prototype usage over windows.
4. Score each prototype by:
   - age
   - usage rate
   - residual quality
   - policy stability
   - recency
5. Mark prototypes as:
   - retain
   - watch
   - refresh
   - prune candidate
6. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 15 recovery table and prototype bank

If files are missing, create a fallback memory-bank simulation.

In [ ]:
recovery_path = RESULTS_DIR / "notebook15_prototype_update_and_recovery.csv"
proto_path = RESULTS_DIR / "notebook15_updated_prototypes.csv"

if recovery_path.exists():
    recovery = pd.read_csv(recovery_path)
    print("Loaded:", recovery_path)
else:
    recovery = None

if proto_path.exists():
    prototypes = pd.read_csv(proto_path)
    print("Loaded:", proto_path)
else:
    prototypes = None

if recovery is None or prototypes is None:
    print("Missing prior outputs; creating fallback prototype memory data.")
    rng = np.random.default_rng(42)
    proto_names = [
        "low_entropy_repeating",
        "sequential_ids",
        "uniform_32bit",
        "zipfian_smallints",
        "clustered_ranges",
        "learned_drift_prototype",
    ]
    prototypes = pd.DataFrame({
        "regime": proto_names,
        "entropy_norm": rng.uniform(0.05, 0.95, len(proto_names)),
        "repetition_ratio": rng.uniform(0.0, 1.0, len(proto_names)),
        "locality_small_delta_ratio": rng.uniform(0.0, 1.0, len(proto_names)),
        "cache_window_reuse_proxy": rng.uniform(0.0, 1.0, len(proto_names)),
        "branch_norm": rng.uniform(0.0, 1.0, len(proto_names)),
        "coherence_score": rng.uniform(0.0, 1.0, len(proto_names)),
        "hardware_pressure_proxy": rng.uniform(0.0, 1.0, len(proto_names)),
    })

    n = 220
    rows = []
    for i in range(n):
        if 90 <= i <= 145:
            dom = "learned_drift_prototype"
        else:
            dom = rng.choice(proto_names[:-1])
        rows.append({
            "window_id": i,
            "new_dominant_prototype": dom,
            "new_residual": abs(rng.normal(0.14 if dom == "learned_drift_prototype" else 0.48, 0.05)),
            "new_drift_score": abs(rng.normal(0.22 if dom == "learned_drift_prototype" else 0.08, 0.05)),
            "updated_policy": "prototype_recovery" if dom == "learned_drift_prototype" else "hybrid",
        })
    recovery = pd.DataFrame(rows)

recovery.head(), prototypes.head()

## Normalize schema

In [ ]:
work = recovery.copy().sort_values("window_id").reset_index(drop=True)

if "new_dominant_prototype" not in work.columns:
    if "updated_dominant_prototype" in work.columns:
        work["new_dominant_prototype"] = work["updated_dominant_prototype"]
    else:
        work["new_dominant_prototype"] = "unknown"

if "new_residual" not in work.columns:
    work["new_residual"] = work.get("reconstruction_residual", 0.0)
if "new_drift_score" not in work.columns:
    work["new_drift_score"] = 0.0
if "updated_policy" not in work.columns:
    work["updated_policy"] = "unknown"

work["new_residual"] = pd.to_numeric(work["new_residual"], errors="coerce").fillna(0.0)
work["new_drift_score"] = pd.to_numeric(work["new_drift_score"], errors="coerce").fillna(0.0)

prototypes = prototypes.copy()
if "regime" not in prototypes.columns:
    prototypes["regime"] = [f"prototype_{i}" for i in range(len(prototypes))]

all_proto_names = list(prototypes["regime"])
if "unknown" in set(work["new_dominant_prototype"]) and "unknown" not in all_proto_names:
    all_proto_names.append("unknown")

print("Prototype count:", len(all_proto_names))
print("Windows:", len(work))

## Compute memory-bank metrics

Each prototype gets:

- first seen / last seen,
- age,
- usage count,
- recency gap,
- mean residual,
- mean drift score,
- policy instability.

In [ ]:
max_window = int(work["window_id"].max())

usage_rows = []
for proto in all_proto_names:
    part = work[work["new_dominant_prototype"] == proto].copy()
    if len(part):
        first_seen = int(part["window_id"].min())
        last_seen = int(part["window_id"].max())
        age = max_window - first_seen + 1
        recency_gap = max_window - last_seen
        usage_count = int(len(part))
        usage_rate = usage_count / max(len(work), 1)
        mean_residual = float(part["new_residual"].mean())
        mean_drift = float(part["new_drift_score"].mean())
        policy_switch_rate = float(part["updated_policy"].ne(part["updated_policy"].shift(1)).mean())
    else:
        first_seen = np.nan
        last_seen = np.nan
        age = max_window + 1
        recency_gap = max_window + 1
        usage_count = 0
        usage_rate = 0.0
        mean_residual = float(work["new_residual"].mean())
        mean_drift = float(work["new_drift_score"].mean())
        policy_switch_rate = 0.0

    usage_rows.append({
        "prototype": proto,
        "first_seen": first_seen,
        "last_seen": last_seen,
        "age": age,
        "recency_gap": recency_gap,
        "usage_count": usage_count,
        "usage_rate": usage_rate,
        "mean_residual": mean_residual,
        "mean_drift_score": mean_drift,
        "policy_switch_rate": policy_switch_rate,
    })

memory = pd.DataFrame(usage_rows)
memory

## Score retention, refresh, and pruning

The memory score rewards:

- usage,
- recency,
- low residual,
- low drift,
- policy stability.

In [ ]:
def norm01(s, invert=False):
    s = pd.Series(s).astype(float)
    lo, hi = s.min(), s.max()
    if hi == lo:
        vals = pd.Series(np.ones(len(s)), index=s.index)
    else:
        vals = (s - lo) / (hi - lo)
    if invert:
        vals = 1.0 - vals
    return vals

memory["usage_score"] = norm01(memory["usage_rate"])
memory["recency_score"] = norm01(memory["recency_gap"], invert=True)
memory["residual_quality_score"] = norm01(memory["mean_residual"], invert=True)
memory["drift_quality_score"] = norm01(memory["mean_drift_score"], invert=True)
memory["policy_stability_score"] = norm01(memory["policy_switch_rate"], invert=True)

memory["memory_stability_score"] = (
    0.30 * memory["usage_score"] +
    0.25 * memory["recency_score"] +
    0.20 * memory["residual_quality_score"] +
    0.15 * memory["drift_quality_score"] +
    0.10 * memory["policy_stability_score"]
).clip(0, 1)

def status(row):
    if row["usage_count"] == 0:
        return "prune_candidate"
    if row["memory_stability_score"] >= 0.70:
        return "retain"
    if row["mean_drift_score"] > memory["mean_drift_score"].quantile(0.75):
        return "refresh"
    if row["memory_stability_score"] < 0.35:
        return "watch"
    return "retain"

memory["memory_status"] = memory.apply(status, axis=1)
memory.sort_values("memory_stability_score", ascending=False)

## Simulate aging and forgetting

Prototype weight decays with recency gap, then receives reinforcement from usage.

In [ ]:
half_life = 50.0

memory["age_decay"] = np.exp(-memory["recency_gap"] / half_life)
memory["reinforced_weight"] = (
    0.65 * memory["age_decay"] +
    0.35 * memory["usage_score"]
).clip(0, 1)

memory["effective_memory_weight"] = (
    0.60 * memory["memory_stability_score"] +
    0.40 * memory["reinforced_weight"]
).clip(0, 1)

def final_action(row):
    if row["memory_status"] == "prune_candidate" and row["effective_memory_weight"] < 0.25:
        return "prune"
    if row["memory_status"] == "refresh":
        return "refresh"
    if row["effective_memory_weight"] < 0.35:
        return "watch"
    return "retain"

memory["memory_action"] = memory.apply(final_action, axis=1)
memory.sort_values("effective_memory_weight", ascending=False)

## Build prototype memory event stream

This creates per-window memory events:

- activate
- reuse
- recover
- monitor

In [ ]:
status_map = memory.set_index("prototype")["memory_action"].to_dict()
stability_map = memory.set_index("prototype")["memory_stability_score"].to_dict()

work["prototype_memory_action"] = work["new_dominant_prototype"].map(status_map).fillna("watch")
work["dominant_prototype_stability"] = work["new_dominant_prototype"].map(stability_map).fillna(0.0)

work["memory_event"] = np.where(
    work["new_dominant_prototype"].ne(work["new_dominant_prototype"].shift(1)),
    "activate",
    "reuse"
)
work.loc[work["prototype_memory_action"].eq("refresh"), "memory_event"] = "recover"
work.loc[work["prototype_memory_action"].eq("watch"), "memory_event"] = "monitor"

work[["window_id", "new_dominant_prototype", "prototype_memory_action", "dominant_prototype_stability", "memory_event"]].head()

## Export memory-bank tables

In [ ]:
csv_path = RESULTS_DIR / "notebook16_prototype_memory_bank_aging.csv"
json_path = RESULTS_DIR / "notebook16_prototype_memory_bank_aging.json"
memory_csv_path = RESULTS_DIR / "notebook16_memory_bank_summary.csv"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)
memory.to_csv(memory_csv_path, index=False)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", memory_csv_path)

## Figure 1 — Prototype memory stability scores

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook16_memory_stability_scores.png"

plot_df = memory.sort_values("memory_stability_score")
plt.figure(figsize=(10, 5))
plt.bar(plot_df["prototype"], plot_df["memory_stability_score"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Memory stability score")
plt.title("Prototype Memory Bank: Stability Scores")
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Effective memory weight

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook16_effective_memory_weight.png"

plot_df = memory.sort_values("effective_memory_weight")
plt.figure(figsize=(10, 5))
plt.bar(plot_df["prototype"], plot_df["effective_memory_weight"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Effective memory weight")
plt.title("Prototype Memory Bank: Aging + Reinforcement")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Prototype usage timeline

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook16_prototype_usage_timeline.png"

labels = sorted(work["new_dominant_prototype"].unique())
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 5))
plt.step(work["window_id"], work["new_dominant_prototype"].map(lab_to_id), where="mid")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Dominant prototype")
plt.title("Prototype Memory Bank: Usage Timeline")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Memory action counts

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook16_memory_action_counts.png"

action_counts = memory["memory_action"].value_counts().rename_axis("action").reset_index(name="count")

plt.figure(figsize=(7, 5))
plt.bar(action_counts["action"], action_counts["count"])
plt.xlabel("Memory action")
plt.ylabel("Prototype count")
plt.title("Prototype Memory Bank: Action Counts")
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Memory event timeline

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook16_memory_event_timeline.png"

event_labels = sorted(work["memory_event"].unique())
event_to_id = {lab: i for i, lab in enumerate(event_labels)}

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], work["memory_event"].map(event_to_id), where="mid")
plt.yticks(list(event_to_id.values()), list(event_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Memory event")
plt.title("Prototype Memory Bank: Event Timeline")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Memory status matrix

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook16_memory_status_matrix.png"

mat_cols = [
    "usage_score",
    "recency_score",
    "residual_quality_score",
    "drift_quality_score",
    "policy_stability_score",
    "memory_stability_score",
    "effective_memory_weight",
]

mat = memory.set_index("prototype")[mat_cols]

plt.figure(figsize=(10, 5))
plt.imshow(mat.values, aspect="auto", vmin=0, vmax=1)
plt.yticks(range(len(mat.index)), mat.index)
plt.xticks(range(len(mat.columns)), mat.columns, rotation=45, ha="right")
plt.colorbar(label="Score")
plt.title("Prototype Memory Bank: Status Matrix")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_16_prototype_memory_bank_aging.md"

summary = {
    "windows": int(len(work)),
    "prototype_count": int(len(memory)),
    "retain_count": int((memory["memory_action"] == "retain").sum()),
    "refresh_count": int((memory["memory_action"] == "refresh").sum()),
    "watch_count": int((memory["memory_action"] == "watch").sum()),
    "prune_count": int((memory["memory_action"] == "prune").sum()),
    "mean_memory_stability_score": float(memory["memory_stability_score"].mean()),
    "mean_effective_memory_weight": float(memory["effective_memory_weight"].mean()),
}

lines = [
    "# Report 16 — Prototype Memory Bank Aging",
    "",
    "This report manages an adaptive prototype memory bank using age, usage, residual quality, drift quality, and policy stability.",
    "",
    "Constraint view:",
    "> adaptive prototype memory needs both recall and forgetting.",
    "",
    "## Generated outputs",
    "",
    f"- Window events CSV: `{csv_path}`",
    f"- Window events JSON: `{json_path}`",
    f"- Memory summary CSV: `{memory_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Memory bank summary",
    "",
    memory.sort_values("effective_memory_weight", ascending=False).to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Retained prototypes explain recent windows with stable residuals and useful policy behavior.",
    "- Refresh candidates are useful but unstable, suggesting prototype update or split.",
    "- Watch candidates are low-confidence memory entries that should not be removed immediately.",
    "- Prune candidates are unused or stale prototypes with weak effective memory weight.",
    "",
    "## Next step",
    "",
    "Notebook 17 can build hierarchical prototype trees: split broad prototypes into child prototypes and merge redundant ones.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook16_prototype_memory_bank_aging_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook16_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_16_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))